# Staging notebook — fit_tp_ospulse

**Do not run this notebook directly.**  
Claude edits cells here; copy individual cells into `fit_tp_ospulse.ipynb` to avoid resetting outputs there.

Each section below is labelled with where it goes in the main notebook.

**Requires `ipympl`** for the interactive bad-channel picker:  
`pip install ipympl`  
Add `%matplotlib widget` to the **imports cell** of the main notebook (replacing `%matplotlib inline`).

---
### [REPLACE: Load session cell — section 1]
Only change from original: `bad_ch_file=None` so all 96 channels are loaded for the picker.

In [ ]:
print(f"Loading {monkey} {date} {site} ...")

session = load_ospulse_session(
    monkey          = monkey,
    date            = date,
    site            = site,
    base_path       = base_path,
    sep_stim_params = "1",
    bad_ch_file     = None,   # ← load all 96 channels; picker handles removal
    ds_factor       = ds_factor,
    remove_dc       = True,
    verbose         = True,
)

fs           = session["fs"]
n_conditions = session["n_conditions"]
n_trials     = session["n_trials"]
n_channels   = session["n_channels"]
n_samples    = session["n_samples"]
conditions   = session["conditions"]

print(f"\nfs={fs:.2f} Hz   n_conditions={n_conditions}   "
      f"n_trials={n_trials}   n_channels={n_channels}   n_samples={n_samples}")

---
### [INSERT: after Load session — section 1b]
**Interactive bad-channel picker**

Plots every electrode at its physical position with the trial-averaged trace.  
Click an electrode to toggle it bad (red) / good (blue).  
Press **Enter** or click the **Done** button when finished, then run the next cell.

In [ ]:
%matplotlib widget
# requires: pip install ipympl
# if this line errors, restart the kernel and re-run from the top

import pickle
from matplotlib.widgets import Button

PICKER_CONDITION   = 0      # condition whose trial-average is shown
electrode_pos_file = r"Z:\Yazdan Lab Members\Grad\fschwock\8 Reach\electrode_positions.pkl"

# ── electrode positions ───────────────────────────────────────────────────────
with open(electrode_pos_file, "rb") as _f:
    _ep_raw = pickle.load(_f)

elec_pos = {int(k): (float(np.asarray(v).ravel()[0]),
                     float(np.asarray(v).ravel()[1]))
            for k, v in _ep_raw.items()}

# ── data: trial average, condition 0, all 96 channels ────────────────────────
mean_trace  = session["conditions"][PICKER_CONDITION]["mean"]   # (96, n_samples)
t_axis      = np.arange(mean_trace.shape[1]) / fs

cond_meta   = session["conditions"][PICKER_CONDITION]
stim_freq   = cond_meta["stim_freq"]
stim_pw     = cond_meta["stim_pw"]

valid_channels = sorted(ch for ch in elec_pos if 0 <= ch < mean_trace.shape[0])
positions      = np.array([elec_pos[ch] for ch in valid_channels])

# ── spatial scaling (identical to detect_bad_channels.py) ────────────────────
def _picker_med_sp(vals):
    u = np.unique(np.round(vals, 6))
    if u.size < 2: return 1.0
    d = np.diff(u); d = d[d > 0]
    return float(np.median(d)) if d.size else 1.0

x_spacing   = _picker_med_sp(positions[:, 0])
y_spacing   = _picker_med_sp(positions[:, 1])
click_tol   = 0.6 * min(x_spacing, y_spacing)

trace_half_w = 0.5 * 0.80 * x_spacing
trace_half_h = 0.5 * 0.35 * y_spacing

t_norm = (t_axis - t_axis.min()) / (t_axis.max() - t_axis.min())
t_norm = t_norm * 2 * trace_half_w - trace_half_w

amp_scale = float(np.nanpercentile(np.abs(mean_trace[valid_channels]), 99)) or 1.0

# ── state: bad_set is read by the apply cell ──────────────────────────────────
bad_set = set()

COLOR_GOOD = "#4fc3f7"
COLOR_BAD  = "#ef5350"

# ── figure ────────────────────────────────────────────────────────────────────
fig_picker, ax_picker = plt.subplots(figsize=(13, 10))
fig_picker.patch.set_facecolor("#1a1a2e")
ax_picker.set_facecolor("#1a1a2e")
plt.subplots_adjust(bottom=0.08)

trace_lines, dot_markers, label_texts = {}, {}, {}

for ch in valid_channels:
    x0, y0 = elec_pos[ch]
    x_tr   = x0 + t_norm
    y_tr   = y0 + (mean_trace[ch] / amp_scale) * trace_half_h

    trace_lines[ch], = ax_picker.plot(x_tr, y_tr,
                                      color=COLOR_GOOD, lw=0.9, alpha=0.85,
                                      picker=False)
    dot_markers[ch], = ax_picker.plot([x0], [y0], "o",
                                      color=COLOR_GOOD, ms=5, zorder=4,
                                      picker=False)
    label_texts[ch]  = ax_picker.text(x0, y0 - 0.42 * y_spacing, str(ch),
                                      ha="center", va="top", fontsize=6,
                                      color=COLOR_GOOD, zorder=5)

status_text = fig_picker.text(0.5, 0.01, "Bad channels: none",
                              ha="center", va="bottom", fontsize=9,
                              color=COLOR_BAD, transform=fig_picker.transFigure)

ax_picker.set_aspect("equal", adjustable="datalim")
ax_picker.set_xlabel("Electrode X (mm)", color="white", fontsize=9)
ax_picker.set_ylabel("Electrode Y (mm)", color="white", fontsize=9)
ax_picker.tick_params(colors="white")
for sp in ax_picker.spines.values():
    sp.set_edgecolor("#444")
ax_picker.set_title(
    f"{monkey}  {date}  {site}  |  cond {PICKER_CONDITION}: "
    f"{stim_freq:.1f} Hz  pw={stim_pw:.4f}\n"
    "Click electrode to toggle bad  |  Done or Enter when finished",
    fontsize=10, color="white", pad=10,
)
ax_picker.grid(True, alpha=0.12, color="white")

_ax_btn = fig_picker.add_axes([0.44, 0.02, 0.12, 0.04])
_btn    = Button(_ax_btn, "Done", color="#2a2a4a", hovercolor="#3a3a6a")
_btn.label.set_color("white")

# ── callbacks ─────────────────────────────────────────────────────────────────
def _update_ch(ch):
    c = COLOR_BAD if ch in bad_set else COLOR_GOOD
    trace_lines[ch].set(color=c, linewidth=1.5 if ch in bad_set else 0.9)
    dot_markers[ch].set(color=c, markersize=7 if ch in bad_set else 5)
    label_texts[ch].set_color(c)

def _refresh_status():
    bl = sorted(bad_set)
    status_text.set_text(
        f"Bad channels (0-idx): {bl}" if bl else "Bad channels: none"
    )
    fig_picker.canvas.draw_idle()

def _on_click(event):
    if event.inaxes is not ax_picker or event.xdata is None:
        return
    dists   = np.linalg.norm(positions - [event.xdata, event.ydata], axis=1)
    nearest = int(np.argmin(dists))
    if dists[nearest] > click_tol:
        return
    ch = valid_channels[nearest]
    bad_set.discard(ch) if ch in bad_set else bad_set.add(ch)
    _update_ch(ch)
    _refresh_status()

def _on_done(_=None):
    bl = sorted(bad_set)
    print(f"Bad channels (0-indexed): {bl}")
    print("Run the next cell to apply and save.")

def _on_key(event):
    if event.key in ("enter", "q"):
        _on_done()

fig_picker.canvas.mpl_connect("button_press_event", _on_click)
fig_picker.canvas.mpl_connect("key_press_event",    _on_key)
_btn.on_clicked(_on_done)
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

---
### [INSERT: after picker — section 1c]
**Apply bad channels + save pkl**

Run after clicking Done above. Reads `bad_set` from the picker.  
Set `SAVE_TO_PKL = True` once you are happy with the selection.

In [ ]:
import pickle

SAVE_TO_PKL = False   # ← set True to write pkl

# read from picker, or fall back to existing pkl, or empty
if 'bad_set' in dir() and bad_set is not None:
    final_bad = sorted(bad_set)
else:
    _pkl_path = PROJECT_ROOT / "tp_python" / "bad_channels" / f"{monkey}_{date}.pkl"
    if _pkl_path.exists():
        with open(_pkl_path, "rb") as _f:
            _ex = pickle.load(_f)
        final_bad = sorted(ch - 1 for ch in _ex.get(site, []))
        print(f"Loaded from pkl: {final_bad} (0-indexed)")
    else:
        final_bad = []
        print("No picker run and no pkl — no channels removed.")

# ── Drop from session in-place ────────────────────────────────────────────────
_n_ch_raw = session["ecog"].shape[2]
_good     = [ch for ch in range(_n_ch_raw) if ch not in set(final_bad)]

session["ecog"]          = session["ecog"][:, :, _good, :]
session["mean"]          = session["mean"][:, _good, :]
session["bad_channels"]  = final_bad
session["good_channels"] = _good
session["n_channels"]    = len(_good)

for _cond in session["conditions"]:
    _cond["ecog"] = _cond["ecog"][:, _good, :]
    _cond["mean"] = _cond["mean"][_good, :]

n_channels = session["n_channels"]
conditions = session["conditions"]
print(f"Bad channels (0-indexed): {final_bad}")
print(f"Kept {n_channels} / {_n_ch_raw} channels.")

# ── Save to pkl ───────────────────────────────────────────────────────────────
if SAVE_TO_PKL:
    _pkl_path = PROJECT_ROOT / "tp_python" / "bad_channels" / f"{monkey}_{date}.pkl"
    _pkl_path.parent.mkdir(parents=True, exist_ok=True)
    _existing: dict = {}
    if _pkl_path.exists():
        with open(_pkl_path, "rb") as _f:
            _existing = pickle.load(_f)
    _existing[site] = sorted(ch + 1 for ch in final_bad)   # 1-indexed
    with open(_pkl_path, "wb") as _f:
        pickle.dump(_existing, _f)
    print(f"Saved → {_pkl_path}  ({site}: {_existing[site]}  1-indexed)")

---
### [INSERT: after apply cell — section 1d]
**Bandpass filter** (brain-wave band)

Filters all conditions' `ecog` in-place before fitting.  
Choose a band, re-run this cell alone to switch bands (no reload needed).  
**Note:** modifies `conditions` in-place — re-run from load if you want unfiltered data.

In [ ]:
from scipy.signal import butter, sosfiltfilt

BANDS = {
    "delta":      (0.5,   4.0),
    "theta":      (4.0,   8.0),
    "alpha":      (8.0,  13.0),
    "beta":       (13.0, 30.0),
    "low_gamma":  (30.0, 60.0),
    "gamma":      (60.0, 120.0),
    "broadband":  (1.0,  200.0),
}

BAND = "broadband"   # ← change to desired band

fmin, fmax = BANDS[BAND]
fmax = min(fmax, fs / 2.0 * 0.95)   # stay below Nyquist

print(f"Band: {BAND}  ({fmin}–{fmax:.1f} Hz)   fs={fs:.1f} Hz   order=4 Butterworth")
sos = butter(4, [fmin, fmax], btype="bandpass", fs=fs, output="sos")

for _cond in session["conditions"]:
    _ecog = _cond["ecog"].astype(np.float64)         # (n_trials, n_ch, n_samples)
    _ecog = sosfiltfilt(sos, _ecog, axis=-1).astype(np.float32)
    _cond["ecog"] = _ecog
    _cond["mean"] = _ecog.mean(axis=0)

session["ecog"] = np.stack([c["ecog"] for c in session["conditions"]])
session["mean"] = np.stack([c["mean"] for c in session["conditions"]])
conditions = session["conditions"]

print(f"Filtered. ecog shape: {session['ecog'].shape}")

---
### [INSERT: after stim inputs — section 2b]
**Spatial trial-averaged trace map** (electrode-position layout)

Traces plotted at physical (x, y) electrode locations. Full trial in gray; stim-on
segment highlighted per-condition in colour. Bad channels absent from the map.
Requires `electrode_positions.pkl`.

In [ ]:
import pickle

PLOT_CONDITIONS    = FIT_CONDITIONS if 'FIT_CONDITIONS' in dir() else list(range(n_conditions))
electrode_pos_file = r"Z:\Yazdan Lab Members\Grad\fschwock\8 Reach\electrode_positions.pkl"

# ── Electrode positions ───────────────────────────────────────────────────────
with open(electrode_pos_file, "rb") as _f:
    _raw_pos = pickle.load(_f)

_elec_pos = {
    int(k): (float(np.asarray(v).ravel()[0]), float(np.asarray(v).ravel()[1]))
    for k, v in _raw_pos.items()
}

# map original channel number → index in bad-ch-removed array
_good_to_idx = {ch: i for i, ch in enumerate(session["good_channels"])}
_valid_ch    = sorted(ch for ch in _elec_pos if ch in _good_to_idx)
_positions   = np.array([_elec_pos[ch] for ch in _valid_ch])

# ── Spatial scaling ───────────────────────────────────────────────────────────
def _med_sp(v):
    u = np.unique(np.round(v, 6))
    if u.size < 2: return 1.0
    d = np.diff(u); d = d[d > 0]
    return float(np.median(d)) if d.size else 1.0

_x_sp = _med_sp(_positions[:, 0])
_y_sp = _med_sp(_positions[:, 1])
_tw   = 0.8  * _x_sp
_th   = 0.20 * _y_sp
_t_norm = np.linspace(-0.5, 0.5, n_samples)

# global 99th-pct scale across all plotted conditions
_all_means = np.stack([
    conditions[c]["mean"][[_good_to_idx[ch] for ch in _valid_ch], :]
    for c in PLOT_CONDITIONS
])
_amp_scale = float(np.nanpercentile(np.abs(_all_means), 99)) or 1.0

# ── Plot ──────────────────────────────────────────────────────────────────────
_cmap = plt.cm.tab10
fig, ax = plt.subplots(figsize=(12, 10))

for ci, c_idx in enumerate(PLOT_CONDITIONS):
    cond   = conditions[c_idx]
    mean_c = cond["mean"]
    color  = _cmap(ci % 10)
    _stim  = inputs[c_idx][0].astype(bool)

    for i, ch in enumerate(_valid_ch):
        x0, y0 = _positions[i]
        trace  = mean_c[_good_to_idx[ch]]
        x_tr   = x0 + _t_norm * _tw
        y_tr   = y0 + (trace / _amp_scale) * _th

        if ci == 0:   # draw gray background trace once
            ax.plot(x_tr, y_tr, color="0.25", lw=0.8, alpha=0.85)

        if _stim.any():
            _lbl = (f"c{c_idx} {cond['stim_freq']:.0f} Hz / "
                    f"{cond['stim_pw']*pw_unit_scale:.3f} s") if i == 0 else None
            ax.plot(x_tr[_stim], y_tr[_stim],
                    color=color, lw=2.0, alpha=0.95, label=_lbl)

for i, ch in enumerate(_valid_ch):
    x0, y0 = _positions[i]
    ax.scatter(x0, y0, s=12, color="tab:blue", zorder=4)
    ax.text(x0, y0 - 0.30 * _y_sp, str(ch),
            ha="center", va="top", fontsize=7, color="tab:blue")

ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("Electrode X (mm)")
ax.set_ylabel("Electrode Y (mm)")
ax.set_title(
    f"Trial-averaged ECoG — {monkey} {date} {site}  conds {PLOT_CONDITIONS}\n"
    f"gray = full trial  |  colour = stim-on segment  |  scale = 99th pct"
)
ax.grid(True, alpha=0.2)
ax.legend(fontsize=8, loc="upper right", framealpha=0.8)

_out_dir = PROJECT_ROOT / "tp_python" / "plots"
_out_dir.mkdir(parents=True, exist_ok=True)
_path = _out_dir / f"spatial_avg_{monkey}_{date}_{site}_conds{'_'.join(map(str,PLOT_CONDITIONS))}.png"
fig.savefig(_path, dpi=150, bbox_inches="tight")
print(f"Saved → {_path}")
plt.show()